# TP 1 — Cancer du sein et régression logistique (correction)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/ai_for_medicine/blob/main/ia-medecine/notebooks/iam-s1-tp-correction.ipynb)

*IA pour la médecine — séance 1.*

    ## Objectif

    La **classification** est une tâche d'apprentissage supervisé où l'on
    prédit une **catégorie** (et non un nombre).

    Dans ce TP, on prédit si une tumeur du sein est **maligne**
    (cancéreuse) ou **bénigne** (non cancéreuse) à partir de mesures des
    noyaux cellulaires. C'est une **classification binaire** : deux
    sorties possibles.

    | | Régression | Classification (ce TP) |
    |---|---|---|
    | **Sortie** | Un nombre réel | Une catégorie (classe) |
    | **Exemple** | Durée de séjour = 6,5 jours | Tumeur = maligne |
    | **Fonction de perte** | MSE | Entropie croisée |
    | **Métriques** | MSE, MAE, R² | Exactitude, précision, rappel, F1 |

    **Méthode** : explorer → visualiser → découper → entraîner → évaluer
    → valider par validation croisée.

    **Comment travailler** : Fichier → *Enregistrer une copie dans Drive*,
    puis exécutez les cellules dans l'ordre (Maj + Entrée) et complétez
    celles marquées `# TODO` en remplaçant les `...`.

## 1. Mise en place

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report,
                             precision_score, recall_score, f1_score)

## 2. Charger et découvrir les données

Le jeu **Breast Cancer Wisconsin** contient des mesures de noyaux
cellulaires issues de biopsies mammaires. Chaque échantillon est
étiqueté **maligne** (classe 0) ou **bénigne** (classe 1) : c'est le
codage de scikit-learn, à garder en tête pour la suite.

### Exercice 1.1 — Charger le jeu de données

Chargez le jeu de données et affichez sa forme, le nom des classes et
les premières lignes.

**Pourquoi** : savoir combien d'échantillons, combien de variables,
et ce que désignent les classes.

*Indice* : `load_breast_cancer(as_frame=True)`, `data.target_names`,
`df.head()`

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame

print(f"Forme : {df.shape}")
print(f"Classes : {data.target_names}")
df.head()

Les 30 variables sont calculées sur une image numérisée d'une
**cytoponction à l'aiguille fine** d'une masse mammaire. Elles
décrivent les noyaux des cellules.

**10 mesures de base** :

- **radius** (rayon) — distance moyenne du centre au contour du noyau
- **texture** — écart-type des niveaux de gris
- **perimeter** (périmètre) et **area** (aire) du noyau
- **smoothness** (régularité) — variation locale du rayon
- **compactness** (compacité) — périmètre² / aire − 1
- **concavity** (concavité) — profondeur des parties concaves du contour
- **concave points** — nombre de parties concaves du contour
- **symmetry** (symétrie) du noyau
- **fractal dimension** — complexité du contour

**Trois variantes par mesure** : `mean` (moyenne sur les noyaux de
l'image), `error` (erreur standard), `worst` (moyenne des trois plus
grandes valeurs).

**Cible** : maligne (0) / bénigne (1).

### Exercice 1.2 — Statistiques descriptives

Affichez les statistiques descriptives des variables.

*Indice* : `df.describe()`

**Question** : les variables ont-elles des échelles comparables ?
Qu'est-ce que cela suggère pour la préparation des données ?

In [ ]:
df.describe()

**Réponse** : non. `mean area` va jusqu'à environ 2 500 alors que
`mean smoothness` reste entre 0,05 et 0,16. Il faudra **mettre les
variables à l'échelle** avant l'entraînement : la régression
logistique y est sensible (partie 6).

### Exercice 1.3 — Équilibre des classes

Comptez les tumeurs malignes et bénignes.

**Pourquoi** : si 95 % des tumeurs étaient bénignes, un modèle qui
répond toujours « bénigne » aurait 95 % d'exactitude tout en étant
inutile. Il faut connaître l'équilibre des classes avant de faire
confiance à l'exactitude.

*Indice* : `y.value_counts()` et `y.value_counts(normalize=True)`

In [ ]:
X = df.drop(columns="target")
y = df["target"]

print(y.value_counts())
print(y.value_counts(normalize=True))

**Réponse** : 357 bénignes (63 %) et 212 malignes (37 %). Les classes
sont raisonnablement équilibrées : l'exactitude a un sens ici, mais on
regardera aussi la précision et le rappel.

## 3. Visualiser les données

En classification, la visualisation montre si les classes sont
**séparables** : peut-on distinguer les deux groupes à l'œil avec
certaines variables ?

### Exercice 2.1 — Distribution d'une variable par classe

Tracez la distribution de `mean radius`, colorée par classe.

**Pourquoi** : si les deux distributions se chevauchent beaucoup, la
variable aide peu à distinguer les classes.

*Indice* : `sns.histplot(data=df, x="mean radius", hue="target", kde=True)`

In [ ]:
sns.histplot(data=df, x="mean radius", hue="target", kde=True, bins=30)
plt.title("Rayon moyen selon la classe (0 = maligne, 1 = bénigne)")
plt.show()

Les distributions diffèrent nettement : les tumeurs malignes ont des
noyaux plus grands. Cette variable sera utile au modèle.

### Exercice 2.2 — Nuage de points de deux variables

Tracez `mean radius` en abscisse, `mean texture` en ordonnée, colorés
par classe.

*Indice* : `sns.scatterplot(data=df, x="mean radius", y="mean texture", hue="target", alpha=0.6)`

In [ ]:
sns.scatterplot(data=df, x="mean radius", y="mean texture",
                hue="target", alpha=0.6)
plt.title("Rayon moyen et texture moyenne (0 = maligne, 1 = bénigne)")
plt.show()

Les deux classes forment deux nuages assez distincts, qui se
chevauchent en partie : elles sont partiellement séparables avec ces
deux variables seulement.

### Exercice 2.3 — Carte des corrélations

Calculez la corrélation de chaque variable avec la cible, gardez les
10 plus fortes en valeur absolue et affichez leur carte de
corrélation (*heatmap*).

**Question** : quelles variables semblent les plus utiles pour
distinguer tumeurs malignes et bénignes ?

In [ ]:
correlations = df.corr()["target"].drop("target").abs()
top = correlations.sort_values(ascending=False).head(10).index.tolist()

plt.figure(figsize=(10, 8))
sns.heatmap(df[top + ["target"]].corr(), annot=True, fmt=".2f",
            cmap="coolwarm", center=0)
plt.tight_layout()
plt.show()
print(correlations.sort_values(ascending=False).head(10))

**Réponse** : les variantes `worst` et `mean` de *concave points*,
*perimeter*, *radius* et *area* (corrélations de 0,7 à 0,8 en valeur
absolue). Ce sont des mesures de taille et de forme des noyaux, des
critères que l'anatomopathologiste regarde aussi.

## 4. La régression logistique

En régression linéaire, on prédit directement un nombre :
$\hat{y} = \theta_0 + \theta_1 x_1 + \dots + \theta_p x_p$.

Pour classer, on veut une **probabilité** entre 0 et 1. La régression
logistique applique la **sigmoïde** à cette combinaison linéaire :

$$P(y=1 \mid x) = \sigma(\theta_0 + \theta_1 x_1 + \dots + \theta_p x_p),
\qquad \sigma(z) = \frac{1}{1 + e^{-z}}$$

**Règle de décision** : si $P(y=1 \mid x) > 0{,}5$, on prédit la classe 1
(bénigne), sinon la classe 0 (maligne).

**Fonction de perte** : l'entropie croisée,

$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i \log(\hat{p}_i)
+ (1-y_i)\log(1-\hat{p}_i)\right]$$

qui pénalise fortement les prédictions fausses faites avec assurance.

### Exercice 3.1 — Découper les données

Découpez en entraînement (80 %) et test (20 %), avec `random_state=42`.

*Indice* : `train_test_split(X, y, test_size=0.2, random_state=42)`

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Entraînement : {X_train.shape[0]} échantillons")
print(f"Test : {X_test.shape[0]} échantillons")

### Exercice 3.2 — Entraîner une régression logistique

Entraînez `LogisticRegression(max_iter=5000)` sur le jeu
d'entraînement et prédisez le jeu de test.

*Indice* : `.fit(X_train, y_train)`, puis `.predict(X_test)`

In [ ]:
model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Prédictions (10 premières) : {y_pred[:10]}")
print(f"Réalité     (10 premières) : {y_test.values[:10]}")

## 5. Évaluer le classifieur

**Exactitude** (*accuracy*) : la part de prédictions correctes. Simple,
mais trompeuse quand les classes sont déséquilibrées.

**Matrice de confusion**, avec la tumeur **maligne comme classe
positive** (c'est elle qu'on cherche à dépister) :

| | Prédit maligne | Prédit bénigne |
|---|---|---|
| **Maligne** | VP (vrai positif) | FN (faux négatif : cancer manqué) |
| **Bénigne** | FP (faux positif : fausse alerte) | VN (vrai négatif) |

| Métrique | Formule | Question |
|---|---|---|
| **Précision** | $\frac{VP}{VP + FP}$ | Parmi les tumeurs prédites malignes, combien le sont ? |
| **Rappel** (sensibilité) | $\frac{VP}{VP + FN}$ | Parmi les tumeurs malignes, combien a-t-on trouvées ? |
| **F1** | $2 \cdot \frac{P \cdot R}{P + R}$ | Moyenne harmonique des deux |

En diagnostic, **le rappel est crucial** : un cancer manqué (faux
négatif) est bien plus grave qu'une fausse alerte, qui mène à des
examens complémentaires.

### Exercice 4.1 — Exactitude

*Indice* : `accuracy_score(y_test, y_pred)`

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f"Exactitude : {acc:.4f}")

### Exercice 4.2 — Matrice de confusion

Affichez la matrice de confusion avec le nom des classes.

*Indice* : `ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=data.target_names)`

**Question** : combien de tumeurs malignes ont été manquées (faux
négatifs) ? Pourquoi est-ce le chiffre important ?

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=data.target_names, cmap="Blues")
plt.title("Matrice de confusion")
plt.show()

**Réponse** : la ligne *malignant* (réalité), colonne *benign*
(prédiction) : 4 des 43 tumeurs malignes du test sont prédites
bénignes. Ce sont
les erreurs les plus graves : une patiente atteinte d'un cancer serait
rassurée à tort.

### Exercice 4.3 — Précision, rappel et F1 à la main

Extrayez VP, VN, FP et FN de la matrice, calculez précision, rappel
et F1 avec les formules, puis vérifiez avec scikit-learn.

*Indice* : `confusion_matrix(y_test, y_pred)` a pour lignes la réalité
et pour colonnes la prédiction, dans l'ordre des classes 0 (maligne)
puis 1 (bénigne). La classe positive étant la maligne (0), il faut
`pos_label=0` dans les fonctions de scikit-learn.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

VP = cm[0, 0]   # maligne prédite maligne
FN = cm[0, 1]   # maligne prédite bénigne
FP = cm[1, 0]   # bénigne prédite maligne
VN = cm[1, 1]   # bénigne prédite bénigne

precision = VP / (VP + FP)
rappel = VP / (VP + FN)
f1 = 2 * precision * rappel / (precision + rappel)
print(f"Précision {precision:.4f}  rappel {rappel:.4f}  F1 {f1:.4f}")
print(f"scikit-learn : {precision_score(y_test, y_pred, pos_label=0):.4f}  "
      f"{recall_score(y_test, y_pred, pos_label=0):.4f}  "
      f"{f1_score(y_test, y_pred, pos_label=0):.4f}")

### Exercice 4.4 — Rapport de classification complet

*Indice* : `print(classification_report(y_test, y_pred, target_names=data.target_names))`

In [ ]:
print(classification_report(y_test, y_pred,
                            target_names=data.target_names))

Le rapport donne précision, rappel et F1 pour **chaque** classe, plus
l'exactitude et deux moyennes (*macro* : moyenne simple des classes ;
*weighted* : pondérée par leurs effectifs). La ligne *malignant*
reprend les valeurs calculées à la main.

## 6. Mettre les variables à l'échelle

La régression logistique minimise l'entropie croisée par un algorithme
d'optimisation itératif, de la famille de la descente de gradient.
Quand les variables ont des échelles très différentes, la surface de
perte est étirée et l'optimisation converge lentement ou mal.

La **standardisation** ramène chaque variable à une moyenne 0 et un
écart-type 1 :

$$x_{\text{standardisé}} = \frac{x - \mu}{\sigma}$$

**À retenir** : $\mu$ et $\sigma$ sont calculés sur le jeu
d'**entraînement** seulement (`fit_transform`), puis appliqués au
test (`transform`).

### Exercice 5.1 — Standardiser et réentraîner

Standardisez avec `StandardScaler`, réentraînez, et comparez
l'exactitude et la matrice de confusion.

**Question** : la mise à l'échelle a-t-elle amélioré les résultats ?
Pourquoi ?

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model_s = LogisticRegression(max_iter=5000)
model_s.fit(X_train_s, y_train)
y_pred_s = model_s.predict(X_test_s)

print(f"Exactitude sans mise à l'échelle : {accuracy_score(y_test, y_pred):.4f}")
print(f"Exactitude avec mise à l'échelle : {accuracy_score(y_test, y_pred_s):.4f}")
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_s, display_labels=data.target_names, cmap="Blues")
plt.show()

**Réponse** : l'exactitude passe de 0,956 à 0,974 (3 erreurs au lieu
de 5 sur 114 tumeurs). Avec `max_iter=5000`, le modèle non standardisé finit par
converger, mais lentement ; sur des variables standardisées,
l'optimisation converge vite et vers un meilleur réglage. Le gain
dépend du jeu de données.

## 7. Validation croisée

Un seul découpage donne une seule estimation, qui dépend du hasard du
découpage. La **validation croisée à 5 blocs** découpe les données en
5 parts, entraîne 5 fois en testant chaque fois sur une part
différente, et donne la moyenne et l'écart-type des 5 scores.

Pour standardiser **dans** chaque tour (sans que le bloc de test
serve à calculer $\mu$ et $\sigma$), on enchaîne la standardisation
et le modèle avec `make_pipeline`.

### Exercice 6.1 — Validation croisée à 5 blocs

*Indice* : `cross_val_score(modele, X, y, cv=5, scoring="accuracy")`

**Question** : le modèle est-il stable d'un bloc à l'autre ?

In [ ]:
modele = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
scores = cross_val_score(modele, X, y, cv=5, scoring="accuracy")

print(f"Scores par bloc : {np.round(scores, 3)}")
print(f"Exactitude = {scores.mean():.4f} ± {scores.std():.4f}")

**Réponse** : l'écart-type est faible (de l'ordre de 0,01 à 0,02) :
les performances varient peu selon le découpage, le modèle est
stable. L'exactitude moyenne est une estimation plus fiable que celle
d'un seul découpage.

## 8. Bilan

| | Régression | Classification (ce TP) |
|---|---|---|
| **Tâche** | Prédire une valeur continue | Prédire une catégorie |
| **Sortie** | Un nombre réel | Une classe (0 ou 1), via une probabilité |
| **Modèle** | Régression linéaire | Régression logistique |
| **Perte** | MSE | Entropie croisée |
| **Métriques** | MSE, MAE, R² | Exactitude, précision, rappel, F1 |

La **méthode** est la même dans les deux cas :

1. **Explorer** les données ;
2. **Visualiser** les relations ;
3. **Découper** en entraînement et test pour une évaluation honnête ;
4. **Entraîner** le modèle adapté ;
5. **Évaluer** avec les bonnes métriques ;
6. **Valider** par validation croisée.

Prochaine étape : le **défi 1**, la même méthode sur des patients de
réanimation.